In [2]:
import pandas as pd
folder_path = r"C:\Users\danis\Desktop\Danish\NVE\glaciers\Mass Balance\Alfotbreen.csv"
alfotbreen_csv= pd.read_csv(folder_path, encoding='cp1252', sep=';')
for col in ["Bw", "Bs", "Ba"]:
    alfotbreen_csv[col] = alfotbreen_csv[col].astype(str).str.replace(",", ".").astype(float)
print(alfotbreen_csv.head())


   Year  GlacierId DateMinPrevYear     DateMax     DateMin ElaPrefix     Ela  \
0  1963       2078      10.11.1962  23.05.1963  22.10.1963       NaN  1270.0   
1  1964       2078      22.10.1963  28.04.1964  04.11.1964       NaN  1165.0   
2  1965       2078      04.11.1964  28.05.1965  22.10.1965       NaN  1105.0   
3  1966       2078      22.10.1965  09.04.1966  18.11.1966         >  1380.0   
4  1967       2078      18.11.1966  29.04.1967  10.12.1967       NaN  1020.0   

    Aar  ElMin  ElMax   Area    Bw    Bs    Ba  Bcalv DataOwner  \
0  37.0    869   1380  4,486  2.52 -3.21 -0.70    NaN       NVE   
1  69.0    869   1380  4,486  2.66 -2.38  0.28    NaN       NVE   
2  83.0    869   1380  4,486  3.75 -3.07  0.68    NaN       NVE   
3   0.0    869   1380  4,486  2.40 -3.93 -1.53    NaN       NVE   
4  94.0    869   1380  4,486  4.43 -3.12  1.30    NaN       NVE   

                                MassBalanceReference  Flag ReanalysisStatus  
0  Kjøllmoen, B. 2016. Reanalysing a g

In [3]:
alfotbreen_csv= alfotbreen_csv.dropna(subset=['Year', 'Bw', 'Bs', 'Ba'])

alfotbreen_df = alfotbreen_csv[["Year", "Bw", "Bs", "Ba"]]
print(alfotbreen_df.head())

   Year    Bw    Bs    Ba
0  1963  2.52 -3.21 -0.70
1  1964  2.66 -2.38  0.28
2  1965  3.75 -3.07  0.68
3  1966  2.40 -3.93 -1.53
4  1967  4.43 -3.12  1.30


### Loop for all stations

In [6]:
import pandas as pd
import os
import glob

# Folder where the original CSVs are stored
folder_path = r"C:\Users\danis\Desktop\Danish\NVE\glaciers\Mass Balance"
output_folder = os.path.join(folder_path, "Cleaned")
os.makedirs(output_folder, exist_ok=True)

csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

for file in csv_files:
    print (f"Processing: {os.path.basename(file)}")

    df= pd.read_csv(file, encoding='cp1252', sep=';')

    for col in ['Bw', 'Bs', 'Ba']:
        df[col]= df[col].astype(str).str.replace(",", ".").astype(float)
        
    df["Time"] = pd.to_datetime(df["Year"].astype(int), format="%Y")  # → gives 01-01-YYYY

    df= df.dropna(subset= ['Time', 'Bw', 'Bs', 'Ba'])
    df_clean= df[['Time', 'Bw', 'Bs', 'Ba']].set_index('Time').sort_index()

    clean_filename= os.path.basename(file).replace('.csv', '_cleaned.csv')
    out_path= os.path.join(output_folder, clean_filename)
    df_clean.to_csv(out_path)

    print(f" Saved: {clean_filename}")

Processing: Alfotbreen.csv
 Saved: Alfotbreen_cleaned.csv
Processing: Austdalsbreen.csv
 Saved: Austdalsbreen_cleaned.csv
Processing: Engabreen.csv
 Saved: Engabreen_cleaned.csv
Processing: Grafjellsbrea.csv
 Saved: Grafjellsbrea_cleaned.csv
Processing: Grasubreen.csv
 Saved: Grasubreen_cleaned.csv
Processing: Hansebreen.csv
 Saved: Hansebreen_cleaned.csv
Processing: Hellstugubreen.csv
 Saved: Hellstugubreen_cleaned.csv
Processing: Langfjordjokelen.csv
 Saved: Langfjordjokelen_cleaned.csv
Processing: Nigardsbreen.csv
 Saved: Nigardsbreen_cleaned.csv
Processing: Rembesdalskåka.csv
 Saved: Rembesdalskåka_cleaned.csv
Processing: Storbreen.csv
 Saved: Storbreen_cleaned.csv


### For converting ERA5 data for each var to seasonal mean

In [1]:
import os
import xarray as xr
import pandas as pd

files= {
    't2m'  : r"C:\Users\danis\Desktop\Danish\ERA5 Data\t2m_djf_fixed.nc",
    'msl'  : r"C:\Users\danis\Desktop\Danish\ERA5 Data\msl.nc",
    'tp'   : r"C:\Users\danis\Desktop\Danish\ERA5 Data\tp_djf_fixed.nc",
    'v10'  : r"C:\Users\danis\Desktop\Danish\ERA5 Data\v10_djf_fixed.nc",
    'u10'  : r"C:\Users\danis\Desktop\Danish\ERA5 Data\u10_djf_fixed.nc",
    'ssrd' : r"C:\Users\danis\Desktop\Danish\ERA5 Data\ssrd_djf_fixed.nc",
}

#predictor_dir = os.path.dirname(t2m_file)

output_dir = r"C:\Users\danis\Desktop\Danish\ERA5 Data\DJF_MEANS"
os.makedirs(output_dir, exist_ok=True)

In [3]:
# DJF mean function
def compute_djf_mean(da):
    # Sort time and keep only DJF months
    da = da.sortby("time")
    da = da.sel(time=da.time.dt.month.isin([12, 1, 2]))

    # Assign each month to the DJF year: Dec goes to next year
    djf_years = da.time.dt.year + (da.time.dt.month == 12).astype(int)
    da.coords["djf_year"] = ("time", djf_years.data)

    # Group by DJF year and average over time
    djf = da.groupby("djf_year").mean("time")

    # Rename time axis properly for PyESD (as datetime)
    djf["djf_year"] = pd.to_datetime(djf["djf_year"].values, format="%Y")
    djf = djf.rename(djf_year="time")

    return djf


# Process and save
for var, filepath in files.items():
    ds = xr.open_dataset(filepath)
    
    # Get the variable
    da = ds[var] if var in ds.data_vars else list(ds.data_vars.values())[0]
    
    # Compute DJF mean
    djf_mean = compute_djf_mean(da)
    djf_mean.name = var
    
    # Fix the time: convert integer years → datetime64 (e.g., 1950 → 1950-01-01)
    djf_mean['time'] = pd.to_datetime(djf_mean['time'].values, format="%Y")
    djf_mean = djf_mean.sel(time=~djf_mean.time.to_index().duplicated())

    # Save with correct format
    output_path = os.path.join(output_dir, f"DJF_MEAN_{var}.nc")
    djf_mean.to_netcdf(output_path)
    
    print(f"Saved: {output_path}")


Saved: C:\Users\danis\Desktop\Danish\ERA5 Data\DJF_MEANS\DJF_MEAN_t2m.nc
Saved: C:\Users\danis\Desktop\Danish\ERA5 Data\DJF_MEANS\DJF_MEAN_msl.nc
Saved: C:\Users\danis\Desktop\Danish\ERA5 Data\DJF_MEANS\DJF_MEAN_tp.nc
Saved: C:\Users\danis\Desktop\Danish\ERA5 Data\DJF_MEANS\DJF_MEAN_v10.nc
Saved: C:\Users\danis\Desktop\Danish\ERA5 Data\DJF_MEANS\DJF_MEAN_u10.nc
Saved: C:\Users\danis\Desktop\Danish\ERA5 Data\DJF_MEANS\DJF_MEAN_ssrd.nc


### For ablation season

In [6]:
import os
import xarray as xr
import pandas as pd

files= {
    't2m'  : r"C:\Users\danis\Desktop\Danish\ERA5 Data\t2m_mjjas_fixed.nc",
    'msl'  : r"C:\Users\danis\Desktop\Danish\ERA5 Data\msl.nc",
    'tp'   : r"C:\Users\danis\Desktop\Danish\ERA5 Data\tp_mjjas_fixed.nc",
    'v10'  : r"C:\Users\danis\Desktop\Danish\ERA5 Data\v10_mjjas_fixed.nc",
    'u10'  : r"C:\Users\danis\Desktop\Danish\ERA5 Data\u10_mjjas_fixed.nc",
    'ssrd' : r"C:\Users\danis\Desktop\Danish\ERA5 Data\ssrd_mjjas_fixed.nc",
}

#predictor_dir = os.path.dirname(t2m_file)

output_dir = r"C:\Users\danis\Desktop\Danish\ERA5 Data\MJJAS_MEANS"
os.makedirs(output_dir, exist_ok=True)

In [7]:
# MJJAS mean function
def compute_mjjas_mean(da):
    # Sort time and keep only DJF months
    da = da.sortby("time")
    da = da.sel(time=da.time.dt.month.isin([5, 6, 7, 8, 9]))

    mjjas_year= da.time.dt.year
    da.coords["mjjas_year"] = ("time", mjjas_year.data)

    # Group by DJF year and average over time
    mjjas = da.groupby("mjjas_year").mean("time")

    # Rename time axis properly for PyESD (as datetime)
    mjjas["mjjas_year"] = pd.to_datetime(mjjas["mjjas_year"].values, format="%Y")
    mjjas = mjjas.rename(mjjas_year="time")

    return mjjas


# Process and save
for var, filepath in files.items():
    ds = xr.open_dataset(filepath)
    
    # Get the variable
    da = ds[var] if var in ds.data_vars else list(ds.data_vars.values())[0]
    
    # Compute DJF mean
    mjjas_mean = compute_mjjas_mean(da)
    mjjas_mean.name = var
    
    # Fix the time: convert integer years → datetime64 (e.g., 1950 → 1950-01-01)
    mjjas_mean['time'] = pd.to_datetime(mjjas_mean['time'].values, format="%Y")
    mjjas_mean = mjjas_mean.sel(time=~mjjas_mean.time.to_index().duplicated())

    # Save with correct format
    output_path = os.path.join(output_dir, f"MJJAS_MEAN_{var}.nc")
    mjjas_mean.to_netcdf(output_path)
    
    print(f"Saved: {output_path}")


Saved: C:\Users\danis\Desktop\Danish\ERA5 Data\MJJAS_MEANS\MJJAS_MEAN_t2m.nc
Saved: C:\Users\danis\Desktop\Danish\ERA5 Data\MJJAS_MEANS\MJJAS_MEAN_msl.nc
Saved: C:\Users\danis\Desktop\Danish\ERA5 Data\MJJAS_MEANS\MJJAS_MEAN_tp.nc
Saved: C:\Users\danis\Desktop\Danish\ERA5 Data\MJJAS_MEANS\MJJAS_MEAN_v10.nc
Saved: C:\Users\danis\Desktop\Danish\ERA5 Data\MJJAS_MEANS\MJJAS_MEAN_u10.nc
Saved: C:\Users\danis\Desktop\Danish\ERA5 Data\MJJAS_MEANS\MJJAS_MEAN_ssrd.nc
